## ***NLP***
- ***NLP is a field of AI that enables machines to process, understand and generate human language using techniques from machine learning, deep learning and linguistics.***

### ***RNN***
- ***RNN remembers information from previous time steps(having vanishing gradient problem).***

### ***LSTM (resolve rnn problem)***
- ***LSTM was designed to solve the long-term dependency problem of vanilla RNNs & it's uses gates to decide what information to keep and updated.***

- ***LSTM is an advanced form of RNN that uses gated memory and a separate cell state to handle long-term dependencies more effectively than vanilla RNN.***

### ***GRU (Alternative of LSTM)***
- ***It is an improved version of the vanilla RNN designed to handle long-term dependencies and reduce the vanishing-gradient problem.***

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("../content/imdb_data.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.columns

Index(['review', 'sentiment'], dtype='object')

In [5]:
sentiment_names = df["sentiment"].unique()
sentiment_names

array(['positive', 'negative'], dtype=object)

In [6]:
df.shape

(50000, 2)

In [7]:
df.isnull().sum()

,0
review,0
sentiment,0


In [8]:
df.drop_duplicates(inplace=True)

In [9]:
df.shape

(49582, 2)

- ***Text Pre-Processing***

In [10]:
## Converting to ;owercase
df["review"] = df["review"].str.lower()

# removing urls
import re
def remove_urls(text):
    text = re.sub(r"http\S+" , "", text)  # (pattern, repl, string) eg - https://www.google.com
    return text

df["review"] = df["review"].apply(remove_urls)

# removing punctuations
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "", text) # A-Z a-z 0-9 \s
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [11]:
## removing HTML
def remove_html(text):
    text = re.sub(r"<.*?>" , "", text)
    return text

df["review"] = df["review"].apply(remove_html)

In [12]:
# remove stopwords
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [13]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopwords(text):
  tokens = word_tokenize(text)
  stop_words = stopwords.words("english")

  for word in tokens:
    if word in stop_words:
      text = text.replace(word,"")
  return text
df["review"] = df["review"].apply(remove_stopwords)

In [14]:
## Stemming
# running -> run
# played -> play

from nltk.stem import PorterStemmer

def stemming(text):
  ps = PorterStemmer()
  stemmed_words = []
  tokens = word_tokenize(text)

  for token in tokens:
    stemmed_token = ps.stem(token)
    stemmed_words.append(stemmed_token)
  return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [15]:
df.sample(10)

,review,sentiment
25545,prvlege wtch mr dentro lst frdi stll shock ts ...,positive
30198,y box 1971 bombbr br sure lke lookng nude wome...,negative
1357,look wrd vewng flm seeng gret cr perm howev sd...,negative
3932,frst becam dssi watchng th move fve mnute caus...,negative
25939,ph boy nmed m m pender wk pr shdi chmney sweep...,positive
48249,bibl tec us love mey root evil love mey led gr...,positive
18731,wfe never got n movew thought t wy sloooowwww ...,negative
19935,never much lked myr move tho pprece t push hol...,negative
46929,one dumbest flm ve ever seen rp f nerli ever t...,negative
14253,first entir script ly improv ddg ftstic illus ...,positive


- ***Encoding***

In [16]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

In [17]:
y = df["sentiment"]

- ***Vectorization***

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df["review"])

- ***Build Dataset & DataLoaders***

In [19]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.3,random_state=42
)

In [20]:
X_train.shape

(34707, 5000)

In [21]:
X_test.shape

(14875, 5000)

In [25]:
import torch
from torch.utils.data import TensorDataset, DataLoader

train_set = TensorDataset(
    torch.from_numpy(X_train.toarray()).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test.toarray()).float(),
    torch.from_numpy(y_test.values).float()
)

train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

#### ***Build Our RNN Model***

In [26]:
import torch.nn as nn
import torch.optim as optim

class RNN(nn.Module):
  def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

  def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [27]:
input_size = X_train.shape[1]
model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [28]:
# Train Our RNN

epochs = 50
for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction

        outputs = model(Xb) # (batch_size, 1)
        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch:{epoch+1}/{epochs} & loss:{loss.item()}")

epoch:1/50 & loss:0.18316662311553955
epoch:2/50 & loss:0.208261638879776
epoch:3/50 & loss:0.4529113173484802
epoch:4/50 & loss:0.3464098274707794
epoch:5/50 & loss:0.1695641130208969
epoch:6/50 & loss:0.6032535433769226
epoch:7/50 & loss:0.31263330578804016
epoch:8/50 & loss:0.3456392288208008
epoch:9/50 & loss:0.2530052661895752
epoch:10/50 & loss:0.2595996558666229
epoch:11/50 & loss:0.143147274851799
epoch:12/50 & loss:0.29968488216400146
epoch:13/50 & loss:0.25052621960639954
epoch:14/50 & loss:0.19307410717010498
epoch:15/50 & loss:0.1469968557357788
epoch:16/50 & loss:0.19559499621391296
epoch:17/50 & loss:0.16048969328403473
epoch:18/50 & loss:0.14763064682483673
epoch:19/50 & loss:0.2418741136789322
epoch:20/50 & loss:0.5126204490661621
epoch:21/50 & loss:0.2838001251220703
epoch:22/50 & loss:0.12796026468276978
epoch:23/50 & loss:0.0884409248828888
epoch:24/50 & loss:0.4061797261238098
epoch:25/50 & loss:0.20570378005504608
epoch:26/50 & loss:0.2503584325313568
epoch:27/50 &

In [29]:
# evaluate metrics
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()
    print(f"accuracy:{correct_vals/tot_vals*100}")

accuracy:84.91428571428571


In [30]:
import torch
torch.save(model.state_dict(), 'rnn_model.pt')
print("model created successfully..✅")

model created successfully..✅


### ***LSTM & GRU Model***

In [41]:
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report
import torch

# LSTM Model Definition
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.5):
        super(LSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)
        self.dropout_layer = nn.Dropout(dropout)

    def forward(self, x):

        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = self.dropout_layer(out[:, -1, :])
        out = self.fc(out)
        return out

In [42]:
# GRU Model
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.5):
        super(GRUModel, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)
        self.dropout_layer = nn.Dropout(dropout)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        out, hn = self.gru(x, h0) # GRU returns only hidden state, no cell state
        out = self.dropout_layer(out[:, -1, :])
        out = self.fc(out)
        return out


In [43]:
# Training and Evaluation
def train_and_evaluate_model(model, train_loader, test_loader, criterion, optimizer, label_encoder, epochs=25):

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for Xb, yb in train_loader:
            optimizer.zero_grad()

            Xb = Xb.unsqueeze(1) # Add sequence length dimension (seq_len=1 for TF-IDF vectors)

            outputs = model(Xb)
            outputs = torch.sigmoid(outputs.squeeze()) # Apply sigmoid for binary classification probability

            loss = criterion(outputs, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch:{epoch+1}/{epochs} &  Average Loss: {total_loss/len(train_loader):.4f}")

    print(f"\n--- Evaluating {model.__class__.__name__} ---")
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        correct_vals = 0
        tot_vals = 0

        for Xb, yb in test_loader:
            Xb = Xb.unsqueeze(1)

            outputs = model(Xb)
            predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float() # Threshold at 0.5 for binary prediction

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

            tot_vals += yb.size(0)
            correct_vals += (predicted == yb).sum().item()

    accuracy = correct_vals/tot_vals
    print(f"Accuracy: {accuracy*100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=label_encoder.classes_))

In [44]:
# Common setup for models
input_size = X_train.shape[1]
epochs_to_run = 30

# LSTM Implementation
print("Initializing LSTM Model")
lstm_model = LSTMModel(input_size, hidden_size=128, num_layers=2, dropout=0.3)
lstm_criterion = nn.BCELoss() # Binary Cross-Entropy Loss for binary classification
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)

# Training and evaluating LSTM
train_and_evaluate_model(lstm_model, train_loader, test_loader, lstm_criterion, lstm_optimizer, le, epochs=epochs_to_run)

# Save LSTM model state dictionary
torch.save(lstm_model.state_dict(), 'lstm_model.pt')
print("LSTM model saved successfully to 'lstm_model.pt' ✅")
print("\n" + "="*50 + "\n")

Initializing LSTM Model
Epoch:1/30 &  Average Loss: 0.3853
Epoch:2/30 &  Average Loss: 0.2618
Epoch:3/30 &  Average Loss: 0.2402
Epoch:4/30 &  Average Loss: 0.2246
Epoch:5/30 &  Average Loss: 0.2054
Epoch:6/30 &  Average Loss: 0.1827
Epoch:7/30 &  Average Loss: 0.1655
Epoch:8/30 &  Average Loss: 0.1473
Epoch:9/30 &  Average Loss: 0.1339
Epoch:10/30 &  Average Loss: 0.1200
Epoch:11/30 &  Average Loss: 0.1075
Epoch:12/30 &  Average Loss: 0.0968
Epoch:13/30 &  Average Loss: 0.0843
Epoch:14/30 &  Average Loss: 0.0780
Epoch:15/30 &  Average Loss: 0.0712
Epoch:16/30 &  Average Loss: 0.0646
Epoch:17/30 &  Average Loss: 0.0582
Epoch:18/30 &  Average Loss: 0.0499
Epoch:19/30 &  Average Loss: 0.0447
Epoch:20/30 &  Average Loss: 0.0386
Epoch:21/30 &  Average Loss: 0.0279
Epoch:22/30 &  Average Loss: 0.0224
Epoch:23/30 &  Average Loss: 0.0215
Epoch:24/30 &  Average Loss: 0.0177
Epoch:25/30 &  Average Loss: 0.0155
Epoch:26/30 &  Average Loss: 0.0133
Epoch:27/30 &  Average Loss: 0.0105
Epoch:28/30 &

In [45]:
# GRU Implementation
print("--- Initializing GRU Model ---")
gru_model = GRUModel(input_size, hidden_size=128, num_layers=2, dropout=0.3)
gru_criterion = nn.BCELoss()
gru_optimizer = optim.Adam(gru_model.parameters(), lr=0.001)

# Training and evaluating GRU
train_and_evaluate_model(gru_model, train_loader, test_loader, gru_criterion, gru_optimizer, le, epochs=epochs_to_run)

# Save GRU model state dictionary
torch.save(gru_model.state_dict(), 'gru_model.pt')
print("GRU model saved successfully to 'gru_model.pt' ✅")

print("\nImplementation of LSTM and GRU with evaluation metrics completed.")

--- Initializing GRU Model ---
Epoch:1/30 &  Average Loss: 0.3729
Epoch:2/30 &  Average Loss: 0.2643
Epoch:3/30 &  Average Loss: 0.2442
Epoch:4/30 &  Average Loss: 0.2298
Epoch:5/30 &  Average Loss: 0.2120
Epoch:6/30 &  Average Loss: 0.1930
Epoch:7/30 &  Average Loss: 0.1767
Epoch:8/30 &  Average Loss: 0.1611
Epoch:9/30 &  Average Loss: 0.1474
Epoch:10/30 &  Average Loss: 0.1366
Epoch:11/30 &  Average Loss: 0.1219
Epoch:12/30 &  Average Loss: 0.1149
Epoch:13/30 &  Average Loss: 0.1022
Epoch:14/30 &  Average Loss: 0.0961
Epoch:15/30 &  Average Loss: 0.0851
Epoch:16/30 &  Average Loss: 0.0752
Epoch:17/30 &  Average Loss: 0.0643
Epoch:18/30 &  Average Loss: 0.0532
Epoch:19/30 &  Average Loss: 0.0420
Epoch:20/30 &  Average Loss: 0.0367
Epoch:21/30 &  Average Loss: 0.0302
Epoch:22/30 &  Average Loss: 0.0263
Epoch:23/30 &  Average Loss: 0.0245
Epoch:24/30 &  Average Loss: 0.0204
Epoch:25/30 &  Average Loss: 0.0192
Epoch:26/30 &  Average Loss: 0.0186
Epoch:27/30 &  Average Loss: 0.0190
Epoch: